# Huấn Luyện AI Model Trích Xuất Tên Hải Quan (NER)

Notebook này được thiết kế để chạy trên Google Colab. 
**Mô hình sử dụng:** `vinai/phobert-base` (Rất tốt cho tiếng Việt)

## Hướng dẫn:
1. Ở Menu Colab, chọn **Runtime -> Change runtime type**, chọn **T4 GPU**.
2. Upload file `training_data.jsonl` (được sinh ra từ file `prepare_data.py`) lên thư mục bên trái của Colab.
3. Nhấn **Runtime -> Run all** để tự động cài đặt và train model.
4. Sau khi xong, thư mục `flavour_ner_model` sẽ chứa model của bạn.

In [ ]:
!pip install transformers datasets seqeval
!pip install accelerate -U
!pip install onnx onnxruntime onnxruntime-tools optimum

In [ ]:
import json
from datasets import Dataset, Features, Sequence, ClassLabel, Value

# 1. Đọc dữ liệu
data_path = "training_data.jsonl"
with open(data_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

data_list = [json.loads(line) for line in lines]

label_list = ["O", "B-PROD", "I-PROD"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

def map_labels(example):
    example["ner_tags"] = [label2id[tag] for tag in example["ner_tags"]]
    return example

dataset = Dataset.from_list(data_list)
dataset = dataset.map(map_labels)

# Chia tập train/val
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print("Train size:", len(dataset['train']))
print("Test size:", len(dataset['test']))

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

model_checkpoint = "vinai/phobert-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx] if label[word_idx] != label2id["B-PROD"] else label2id["I-PROD"])
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
from transformers import DataCollatorForTokenClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

data_collator = DataCollatorForTokenClassification(processing_class=tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

args = TrainingArguments(
    "flavour_ner_model",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# Lưu lại mô hình chuẩn
trainer.save_model("flavour_ner_model_final")
print("Đã lưu mô hình Pytorch thành công!")

## (Tuỳ chọn) Chuyển đổi Model sang định dạng ONNX
ONNX giúp model siêu nhẹ, không cần thư viện PyTorch khi tích hợp vào file .exe

In [ ]:
!optimum-cli export onnx --model flavour_ner_model_final --task token-classification flavour_ner_onnx
print("Hoàn tất convert sang ONNX! Bạn hãy tải thư mục 'flavour_ner_onnx' về máy.")